# CLOY Processing

This notebook is for you to work with the CLOY data folder.

In [13]:
import json
import math
from pathlib import Path
from textblob import TextBlob, Word 
!pip3 install scipy
from scipy.sparse import csr_matrix
import numpy as np
!pip3.install pandas 
import pandas as pd 


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
zsh:1: command not found: pip3.install


In [14]:
skip_text = {"[deleted]", "[removed]", ""}
remove_prepositions = False 


In [15]:
# loading posts
titles = {}
thread_info = {}
shown = False
for path in sorted(Path("CLOY").glob("cloy_submissions_*.jsonl")):
    with open(path, encoding="utf-8-sig") as f:
        for line in f:
            if not line.strip():
                continue
            post = json.loads(line)

            # viewing one post
            if not shown:
                print(post)
                shown = True

            # indexing into different keys of post based on previous test 
            post_id = int(post["post_id"], 36)
            titles[post_id] = post["title"]
            thread_info[post_id] = [post["title"]]
            body = post.get("selftext") or ""
            if body.strip() not in skip_text:
                thread_info[post_id].append(body)

# viewing thread_info and titles
print(len(thread_info), "threads")
print(len(titles), "titles")
print(list(titles.items())[:5])

first_id = list(thread_info.keys())[0]
print(first_id, thread_info[first_id])

# viewing one comment to try 
shown = False
for path in sorted(Path("CLOY").glob("cloy_comments_*.jsonl")):
    with open(path, encoding="utf-8-sig") as f:
        for line in f:
            if line.strip():
                print(json.loads(line))
                shown = True
                break
    if shown:
        break

{'timestamp': 1571646432, 'author': '[deleted]', 'subreddit': 'CrashLandingOnYou', 'title': 'Crash Landing On You has been created', 'selftext': '[deleted]', 'url': 'https://www.reddit.com/r/CrashLandingOnYou/comments/dkxewd/crash_landing_on_you_has_been_created/', 'score': 1, 'num_comments': 0, 'permalink': 'https://reddit.com/r/CrashLandingOnYou/comments/dkxewd/crash_landing_on_you_has_been_created/', 'post_id': 'dkxewd', 'full_id': None, 'flair': None}
3066 threads
3066 titles
[(821211565, 'Crash Landing On You has been created'), (932986619, 'ok, let’s get this started! What are your thoughts about the show?'), (935298279, 'Find someone who’ll make you noodles just like Ri Jeong Hyeok-ssi'), (936141897, 'Captain Ri’s squad! Who’s your favorite?'), (939154123, 'Why I think we all need a second season of CLOY. (Spoilers)')]
821211565 ['Crash Landing On You has been created']
{'comment_id': 'fk0rooq', 'post_id': 't3_ffh52z', 'parent_id': 't3_ffh52z', 'subreddit': 'CrashLandingOnYou', 

In [16]:
# adding every comment to its thread
matched = 0
unmatched = 0
comment_info = {}

for path in sorted(Path("CLOY").glob("cloy_comments_*.jsonl")):
    with open(path, encoding="utf-8-sig") as f:
        for line in f:
            if not line.strip():
                continue
            comment = json.loads(line)

            # saving every comment 
            comment_info[comment["comment_id"]] = comment

            # strip the t3_ 
            post_id = comment["post_id"]
            if post_id.startswith("t3_"):
                post_id = post_id[3:]
            post_id = int(post_id, 36)     

            # get the comment's text
            body = comment.get("comment_body") or ""

            # add the text to its thread, with if statement for if thread exists 
            if post_id in thread_info:
                matched += 1
                if body.strip() not in skip_text:
                    thread_info[post_id].append(body)
            else:
                unmatched += 1

#testing 
print(list(comment_info.values())[:3])

    


[{'comment_id': 'fk0rooq', 'post_id': 't3_ffh52z', 'parent_id': 't3_ffh52z', 'subreddit': 'CrashLandingOnYou', 'author': 'chimichangarolls', 'comment_body': "i'm so emotionally drained and exhausted from being in awe of this show even though i finished watching 2 weeks ago. my brain doesn't work cuz i'm thinking about CLOY and my favorite part of it other than the couple is that the women have really 3D roles... like Seri can cry nonstop but you still know she a boss.", 'created_utc': 1583764623, 'score': 19, 'ups': None, 'downs': None, 'controversiality': 0, 'gilded': 0, 'total_awards': 0, 'distinguished': None, 'edited': False}, {'comment_id': 'fk1c27d', 'post_id': 't3_ffh52z', 'parent_id': 't1_fk0rooq', 'subreddit': 'CrashLandingOnYou', 'author': 'bruisedlee123', 'comment_body': 'Did you have a favorite episode? Mine would have to be ep. 9 ☺️', 'created_utc': 1583776621, 'score': 2, 'ups': None, 'downs': None, 'controversiality': 0, 'gilded': 0, 'total_awards': 0, 'distinguished': N

In [17]:
# lemmatize
#created a dictionary for the cleaned version 
docs = {}
#trying to access for words 
#trying first entry 
part = list(thread_info.items())[0]
print(part)
post_id, words = part
print (post_id)
print(words)
blob = TextBlob(" ".join(words))
for word, tag in blob.tags:
    print(word, tag)

for post_id, words in thread_info.items():
    blob = TextBlob(" ".join(words))
    lemmas = []
    for word, tag in blob.tags:
        #making all lowercase 
        word = word.lower()
        #print(word)
        if remove_prepositions and word in stop_words:
            continue
        #verbs 
        if tag.startswith("V"):
            part_speech = "v"
        #adjective 
        elif tag.startswith("J"):
            part_speech = "a"
        #adverb
        elif tag.startswith("R"):
            part_speech = "r"
        #noun 
        else:
            part_speech = "n"
        lemmas.append(Word(word).lemmatize(part_speech))
    docs[post_id] = TextBlob(" ".join(lemmas))

print(list(docs.values())[:3])

(821211565, ['Crash Landing On You has been created'])
821211565
['Crash Landing On You has been created']
Crash NNP
Landing VBG
On IN
You PRP
has VBZ
been VBN
created VBN
[TextBlob("crash land on you have be create"), TextBlob("ok let ’ s get this started what be your thought about the show i haven ’ t see a kdrama for the long time and it be actually my dad who choose to stream cloy on netflix he stop watch after the second episode but i go on after immediately fall in love with the lead hyun bin be just so charismatic a capt ri jeong hyeok and son ye-jin be perfect a yoon se-ri she ’ s definitely get the rom-com and drama genre down pat i also love capt ri ’ s squad most especially pyo chi su ☺️ we can probably talk about individual character next time cloy tackle politics drama and deception family issue friendship and best of all pure unconditional love love bring about by fate all in one highly entertaining and well-written package i ’ m planning on do a rewatch soon when i get a

In [18]:
#collecting the unique words into list 
vocab = sorted({term for blob in docs.values() for term in blob.word_counts})
index = {term: i for i, term in enumerate(vocab)}

In [24]:
# vector
# order for thread to go into row
ids = list(docs)
rows, columns, counts = [], [], []
for position, post_id in enumerate(ids):
    for word, count in docs[post_id].word_counts.items():
        # skips words with a count of 0
        if count:
            rows.append(position)
            columns.append(index[word])
            counts.append(count)
vectors = csr_matrix((counts, (rows, columns)), shape=(len(ids), len(vocab)), dtype=float)

In [22]:
#cosine similarity 
#calculating for one value 
#picking two threads by their position in ids
first = 0
second = 1
print( ids[first], ids[second])

#getting each thread's row from vectors 
a = vectors[first].toarray().ravel()
b = vectors[second].toarray().ravel()

# dot product 
dot_product = np.dot(a, b)

#magnitude of each vector
magnitude_a = np.sqrt(np.sum(a * a))
magnitude_b = np.sqrt(np.sum(b * b))

#cosine similarity
cosine = dot_product / (magnitude_a * magnitude_b)

print(dot_product)
print(magnitude_a, magnitude_b)
print(cosine)

#all 
magnitude = np.sqrt(vectors.multiply(vectors).sum(axis=1)).A1
magnitude[magnitude == 0] = 1
calculation = (vectors @ vectors.T).toarray() / np.outer(magnitude, magnitude)
table = pd.DataFrame(calculation, index=ids, columns=ids)
table.to_csv("cloy_similarity.csv", float_format="%.4f")

821211565 932986619
98.0
2.6457513110645907 174.4161689752415
0.2123686041983997


In [34]:
def top_5_similar(thread_id, n=5):
    key_1 = int(thread_id, 36)
    scores = table[key_1].drop(key_1).sort_values(ascending=False).head(n)
    print(f"{thread_id}: {titles[key_1]}\n")
    for key_2, score in scores.items():
        other_id = np.base_repr(key_2, 36).lower()
        print(f"{other_id}  {titles[key_2]}  ({score:.3f})")

top_5_similar("ffh52z")

ffh52z: ok, let’s get this started! What are your thoughts about the show?

i42ez5  How come you started watching it?  (0.954)
qg3ups  KDrama Virgin hit by CLOY tsunami  (0.950)
1ci4cmh  I have just finished binge-watching Crash Landing On You. I am overwhelmed.  (0.947)
hp9elr  post-series depression  (0.946)
13milt2  I binge-watched Crash Landing on you and now I just can't stop thinking about it.  (0.944)
